
# Getting Started: Daily & Monthly Flux Footprints

This notebook shows how to use **`ffp_daily_monthly_helper.py`** to:
1) load AmeriFlux half‑hourly data,  
2) compute an xarray-based footprint climatology,  
3) summarize to daily/monthly periods (optionally ET‑weighted), and  
4) export **80%** source‑area contours to a GeoPackage or rasters to GeoTIFF.

References: `ffp_daily_monthly_helper.py`【8†source】 and `ffp_xr.py`【9†source】.



## Requirements

This workflow uses: `numpy`, `pandas`, `xarray`, `matplotlib`, and for exports `geopandas`, `pyproj`, `shapely`, `rasterio`.


In [2]:

# --- Imports ---
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

from fluxfootprints import (
    load_config,
    load_amf_df,
    build_climatology,
    summarize_periods,
    export_contours_gpkg,
    export_rasters_geotiff,
    export_contour_stats_csv,
)


ImportError: cannot import name 'load_config' from 'fluxfootprints' (c:\Users\paulinkenbrandt\.conda\envs\py313\Lib\site-packages\fluxfootprints\__init__.py)


## 1) Set Paths

Update these to point at your AmeriFlux config (`.ini`) and half‑hourly `.csv`.


In [ ]:

ini_path = Path("US-CRT_config.ini")
csv_path = Path("AMF_US-CRT_BASE_HH_3-5_abb.csv")

assert ini_path.exists(), f"Config not found: {ini_path}"
assert csv_path.exists(), f"CSV not found: {csv_path}"

out_dir = Path("ffp_outputs")
out_dir.mkdir(parents=True, exist_ok=True)



## 2) Load Configuration & Data
`load_config` parses a minimal INI for site metadata and column mappings.  
`load_amf_df` reads the CSV, parses timestamps, sets the index to time, and replaces missing value sentinels【8†source】.


In [ ]:

cfg = load_config(str(ini_path))
cfg


In [ ]:

df = load_amf_df(str(csv_path), cfg)
display(df.head())
print("Time span:", df.index.min(), "→", df.index.max(), "| rows:", len(df))



## 3) Build the Footprint Climatology
`build_climatology` renames expected AMF columns (WD, WS, USTAR, MO_LENGTH, V_SIGMA) to the solver’s names and runs the **xarray-based** climatology (`ffp_xr.ffp_climatology_new.run()`), filling `clim.f_2d` with per‑timestep footprints【8†source】【9†source】.


In [ ]:

clim = build_climatology(
    df,
    crop_height=0.2,
    atm_bound_height=2000.0,
    inst_height=2.5,
    dx=10.0, dy=10.0,
    domain=(-500.0, 500.0, -500.0, 500.0),  # smaller domain for a quick start
)
clim



## 4) Summarize to Daily / Monthly
- `summarize_periods` normalizes each time slice so that the sum over x,y = 1 (optional), then computes:
  - **Daily/Monthly means**, and
  - **ET‑weighted** versions using ET derived from LE (mm/hr = LE / 680.6)【8†source】.


In [ ]:

summaries = summarize_periods(
    clim,
    df,
    et_source="LE",            # use LE (W/m^2) to derive ET weights
    daily=True,
    monthly=True,
    normalize_each_time=True,
)
summaries



## 5) Quick Visualization

Plot the first available **daily mean** footprint.  
(Uses `matplotlib`; ensure you keep a single plot per figure and default colors.)


In [ ]:

# pick the first day with data
da = summaries.f_daily_mean.isel(time=0)
plt.figure(figsize=(6, 5))
im = plt.imshow(da.values, origin="lower",
                extent=[float(clim.x.min()), float(clim.x.max()),
                        float(clim.y.min()), float(clim.y.max())])
plt.colorbar(im, label="Normalized footprint")
plt.title("Daily Mean Footprint (first day)")
plt.xlabel("x (m)"); plt.ylabel("y (m)")
plt.show()



## 6) Export 80% Contours to a GeoPackage

Writes layers like: `daily_mean_r80`, `monthly_etw_r80`, etc.  
Contours are generated with a robust alternative to `plt.contour` that can use `skimage` or `rasterio`【8†source】.


In [ ]:

gpkg_path = out_dir / "footprints_80pct.gpkg"
gpkg_written = export_contours_gpkg(
    clim,
    summaries,
    df=df,
    station_lat=cfg["station_latitude"],
    station_lon=cfg["station_longitude"],
    gpkg_path=str(gpkg_path),
    crs_out="auto",            # chooses a suitable UTM
    levels=(0.8,),             # export the 80% source-area contour
    contour_method="auto",
)
print("GeoPackage written to:", gpkg_written)



## 7) Export GeoTIFF Rasters

Each time slice (daily/monthly) is written as a separate `.tif` with correct georeferencing around the tower origin【8†source】.


In [ ]:

tif_dir = out_dir / "rasters"
export_rasters_geotiff(
    clim,
    summaries,
    station_lat=cfg["station_latitude"],
    station_lon=cfg["station_longitude"],
    out_dir=str(tif_dir),
    which=("daily_mean", "monthly_etw"),
    prefix="ffp",
)
tif_dir



## 8) Export Contour Stats to CSV

Creates a compact CSV with area (ha) and centroid (lat/lon) for each contour and time slice【8†source】.


In [ ]:

csv_path = out_dir / "contour_stats.csv"
export_contour_stats_csv(
    df,
    clim,
    summaries,
    station_lat=cfg["station_latitude"],
    station_lon=cfg["station_longitude"],
    csv_path=str(csv_path),
    levels=(0.8,),
)
print("Stats CSV saved to:", csv_path)
pd.read_csv(csv_path).head()



## Tips & Troubleshooting

- If you see missing column errors, verify your CSV has the expected fields or update the INI to map the correct names. The helper expects AMF-like columns (e.g., `WD`, `WS`, `USTAR`, `MO_LENGTH`, `V_SIGMA`) and renames them internally【8†source】.
- ET weighting converts LE (W/m²) to mm/hr using `LE / 680.6`【8†source】.
- For exports, ensure `geopandas`, `shapely`, `pyproj`, and `rasterio` are installed.
- To change source-area levels, pass `levels=(0.5, 0.8)` to the export functions.
- To use a fixed CRS (instead of UTM auto), pass `crs_out=EPSG_CODE`.
